In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
from collections import Counter
import time
import json

In [2]:
with open("headers.json", "r") as f:
    headers_dict_1 = json.load(f)

In [3]:
with open("headers_2.json", "r") as f:
    headers_dict_2 = json.load(f)

In [4]:
csv_entries = pd.read_csv('construction_entries_todo.csv', encoding ='latin1', names = ['company names'])

In [5]:
def url_search_term(business_name: str):
    business_name = business_name.replace(",", "")
    business_name = business_name.replace(".", "")
    url_st = business_name.upper().replace(' ', '-')
    return url_st

In [47]:
r = requests.get(url,headers=header_4, timeout=10)
print(str(r.status_code))
soup = BeautifulSoup(r.content)

404


In [6]:
later_entries = csv_entries.iloc[1198:,:]
later_entries['company names'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

1198                                   CASA MESTRET, S.L.
1199                                 CESAR GONZALEZ, S.L.
1200    PREFABRICADOS Y POSTES DE HORMIGON, S.A. (PREP...
1201              TRANSPORTES Y EXCAVACIONES RIBERA, S.A.
1202              EXCAVACIONES Y TRANSPORTES CEREZO, S.A.
                              ...                        
2414                          FERRETERIA ALMATRICHE, S.L.
2415                              ELECTRO SHOW ROOM, S.L.
2416                   GRUPOTEC SERVICIOS AVANZADOS, S.A.
2417                                     GUADACORTE, S.A.
2418                                     TRANSGRUMA, S.A.
Name: company names, Length: 1221, dtype: object

In [55]:
def get_contact_info(df: pd.DataFrame, n: int):
    """
    This company scrapes info of companies for which links were found.

    Parameters
    ----------
    'df' : pd.DataFrame
    """
    phonenumbers_found = []
    urls_found = []
    emails_found = []

    for i, link in enumerate(tqdm(df['found_links'])):
        r = requests.get(link,headers=header_3, timeout=10)
        if r.status_code == 429:
            print(f'The current index is {i} and error code 429.')
            break
        if r.status_code != 200:
            print(str(r.status_code))
        soup = BeautifulSoup(r.content)
        try:
            found_url = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline'))[0]['href']
        except:
            found_url='not found'
        urls_found.append(found_url)
        try:
            found_email = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline email'))[0]['href'].split('?')[0]
        except:
            found_email = 'not found'
        emails_found.append(found_email)
        try:
            found_phone = soup.find_all('span', class_=lambda value: value and value.startswith('text-bodytext-m text-neutrals-700 md:block hidden'))[0].text
        except:
            found_phone = 'not found'
        phonenumbers_found.append(found_phone)
        time.sleep(n)

    info_dict = {'phones': phonenumbers_found, 'urls': urls_found, 'emails': emails_found}
    print(f'The cooldown this time was {n} seconds.')
    return info_dict

In [57]:
contact_info = get_contact_info(entries_infoadded, n=3)

100%|██████████| 176/176 [10:13<00:00,  3.48s/it]

The cooldown this time was 3 seconds.


In [155]:
r = requests.get(test_link,headers=header_3, timeout=10)
print(str(r.status_code))

soup = BeautifulSoup(r.content)

200


In [7]:
def business_search_rotate(csv_entries: pd.DataFrame, n: float, header_dict: dict):
    """
    This function searches the baseurl site for the business names given in the DataFrame csv_entries.
    
    Parameters
    ----------
    """
    link_list = []
    baseurl = 'https://empresite.eleconomista.es/Actividad/'
    header_index = 0

    for i, company in enumerate(tqdm(list(csv_entries['company names']))):
        search_term = url_search_term(company)
        search_url = baseurl + search_term + '/'
        mini_header_dict = {
            'User-Agent' : header_dict[str(header_index)]
        }
        try:
            r = requests.get(search_url,headers=mini_header_dict, timeout=30)
            soup = BeautifulSoup(r.content)
            # print(company)
            # print(str(r.status_code))
            try:
                if str(r.status_code)== '200':
                    # first_result = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0].text
                    # first_result = first_result.capitalize().split(' ')[0]
                    # company_name_start = company.split(' ')[0]
                    
                    # if first_result.split(' ')[0]==company.split(' ')[0]:
                    url_results = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0]['href']
                    link_list.append(url_results)
                elif str(r.status_code)== '429':
                    header_index = header_index + 1
                    try:
                        r = requests.get(search_url,headers=mini_header_dict, timeout=30)
                        soup = BeautifulSoup(r.content)
                        url_results = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0]['href']
                        link_list.append(url_results)
                    except:
                        link_list.append('reset but result issue')
                else:
                    link_list.append('possible 404 not found')
            except IndexError:
                link_list.append('no results found')
            time.sleep(n)
        except:
            link_list.append('requests issue')

    print(f'The cooldown this time was {n} seconds.')
    return link_list

In [8]:
link_list = business_search_rotate(later_entries, n = 2.5, header_dict=headers_dict_1)

  0%|          | 0/1221 [00:00<?, ?it/s]

100%|██████████| 1221/1221 [1:04:48<00:00,  3.18s/it]

The cooldown this time was 2.5 seconds.


In [12]:
Counter(link_list)

Counter({'possible 404 not found': 420,
         'https://empresite.eleconomista.es/NOVELEC-GIRONES.html': 7,
         'reset but result issue': 4,
         'https://empresite.eleconomista.es/SPLITMANIA-SEVILLA.html': 2,
         'https://empresite.eleconomista.es/COMERCIAL-BAGES-RIVERO.html': 2,
         'https://empresite.eleconomista.es/CASA-MESTRET.html': 1,
         'https://empresite.eleconomista.es/TRANSPORTES-EXCAVACIONES-RIBERA.html': 1,
         'https://empresite.eleconomista.es/EXCAVACIONES-TRANSPORTES-CEREZO.html': 1,
         'https://empresite.eleconomista.es/PEDRO-LUIS-DARIAS.html': 1,
         'https://empresite.eleconomista.es/FORJADOS-AZUCENA.html': 1,
         'https://empresite.eleconomista.es/MIRAMUNT-VILAMU.html': 1,
         'https://empresite.eleconomista.es/MADRID-RAMOS-SIERRA.html': 1,
         'https://empresite.eleconomista.es/BORTUBO.html': 1,
         'https://empresite.eleconomista.es/FORJADOS-MARFE.html': 1,
         'https://empresite.eleconomista.es/B

In [15]:
def cleanup_found_links_rotate(df: pd.DataFrame, link_list: list) -> None:
    """
    This function adds urls to company listings that were found and removes rows of companies which were not found listed.

    Parameters
    ----------
    'df' : pd.DataFrame
        DataFrame of companies from csv import.
    'link_list' : list of links found from 'business_search' 
    """
    df = df.copy()
    df.reset_index(drop=True,inplace=True)
    df['found_links'] = pd.DataFrame(link_list, columns=['found_links'])
    excluded_rows = df[(df['found_links']=='possible 404 not found') | (df['found_links']=='reset but result issue') | (df['found_links']=='429 issue') | (df['found_links']=='requests issue')].index
    df.drop(index=excluded_rows, inplace=True)
    df = df.dropna(subset='found_links')
    df.reset_index(drop=True,inplace=True)

    return df

In [16]:
entries_infoadded = cleanup_found_links_rotate(later_entries, link_list)

In [17]:
def get_contact_info_rotate(df: pd.DataFrame, n: int, header_dict: dict):
    """
    This company scrapes info of companies for which links were found.

    Parameters
    ----------
    'df' : pd.DataFrame
    """
    phonenumbers_found = []
    urls_found = []
    emails_found = []
    header_index = 0

    for i, link in enumerate(tqdm(df['found_links'])):
        mini_header_dict = {
            'User-Agent' : header_dict[str(header_index)]
        }
        r = requests.get(link,headers=mini_header_dict, timeout=20)
        if r.status_code == 429:
            header_index = header_index + 1
            r = requests.get(link,headers=mini_header_dict, timeout=20)
        if (r.status_code != 200) & (r.status_code != 429):
            print(str(r.status_code))
        soup = BeautifulSoup(r.content)
        try:
            found_url = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline'))[0]['href']
        except:
            found_url='not found'
        urls_found.append(found_url)
        try:
            found_email = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline email'))[0]['href'].split('?')[0]
        except:
            found_email = 'not found'
        emails_found.append(found_email)
        try:
            found_phone = soup.find_all('span', class_=lambda value: value and value.startswith('text-bodytext-m text-neutrals-700 md:block hidden'))[0].text
        except:
            found_phone = 'not found'
        phonenumbers_found.append(found_phone)
        time.sleep(n)

    info_dict = {'phones': phonenumbers_found, 'urls': urls_found, 'emails': emails_found}
    print(f'The cooldown this time was {n} seconds.')
    return info_dict

In [ ]:
contact_info = get_contact_info_rotate(entries_infoadded, n=3, header_dict=headers_dict_2)

  1%|▏         | 11/797 [00:38<45:53,  3.50s/it]

In [65]:
def add_found_info(df: pd.DataFrame, contact_info_dict: dict)-> None:
    """
    compiles phone numbers, emails, urls into df and fixes some formatting issues.

    Parameters
    ----------
    """
    phone_list = contact_info_dict['phones']
    url_list = contact_info_dict['urls']
    email_list = contact_info_dict['emails']
    
    df['phones'] = pd.DataFrame(phone_list, columns=['phones'])
    df['urls'] = pd.DataFrame(url_list, columns=['urls'])
    df['emails'] = pd.DataFrame(email_list, columns=['emails'])

    # Format corrections:
    df['urls'] = df['urls'].apply(lambda x: x[2:] if x.startswith('//')==True else x)
    df['emails'] = df['emails'].apply(lambda x: x.split(':')[1] if x.startswith('mailto')==True else x)
    df.drop(columns=['found_links'],inplace=True)

    # Removing companies w no info:
    no_info_indices = entries_infoadded.loc[(entries_infoadded['emails']=='not found') & (entries_infoadded['phones']=='not found') & (entries_infoadded['urls']=='not found')].index
    df.drop(index = no_info_indices, inplace=True)
    entries_infoadded.reset_index(drop=True, inplace=True)
    return None

In [66]:
add_found_info(df=entries_infoadded, contact_info_dict=contact_info)

KeyError: "['found_links'] not found in axis"

In [76]:
entries_infoadded.to_csv('construction_leads_4.csv')